# Multi-Asset Scenario Engine for Joint US Equity-Rates Horizon Repricing (v6)

This version broadens the equity universe beyond a narrow tech basket and makes the **equity-premium block modular**. The notebook now supports multiple ERP engines, including a **statistical latent factor**, an **observed market-proxy engine**, and an **observable-factor APT engine** based on daily Fama-French / Carhart-style factors.

**v6** keeps the structure and the section flow unchanged but applies a set of methodological corrections — a unified risk-free convention, additive factor treatment, a richer Treasury curve, a vectorized simulator, idiosyncratic-noise re-injection, an exact min-CVaR LP, an out-of-sample portfolio check, VAR stability/usefulness diagnostics, declared bond-pricing assumptions, and a substantially expanded validation suite with a walk-forward VaR backtest. The specific changes are listed in the final section.



## Research positioning

The notebook is now structured as a **cross-asset scenario engine under the physical measure** with three explicit layers:

1. a **risk-free curve factor generator**,
2. a **plug-and-play equity-premium engine**,
3. an **idiosyncratic residual block** compressed through PCA.

The goal is not point forecasting, but **joint one-week scenario generation**, **instrument-level horizon repricing**, and **portfolio tail-risk analysis**. The current implementation also lets the user **choose the ERP simulation engine**.


In [ ]:
# Core imports and global configuration
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import yfinance as yf
from pandas_datareader import data as web

from sklearn.decomposition import PCA
from statsmodels.tsa.api import VAR

from scipy import sparse
from scipy.optimize import linprog
from scipy.stats import chi2

from pandas.tseries.offsets import BDay

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")
plt.rcParams["figure.figsize"] = (10, 6)

# Broad US equity prototype universe (multi-sector)
TICKERS = ["MSFT", "JPM", "XOM", "JNJ", "PG", "HD", "CAT", "NEM", "PLD", "DUK"]
SECTOR_MAP = {
    "MSFT": "Information Technology",
    "JPM": "Financials",
    "XOM": "Energy",
    "JNJ": "Health Care",
    "PG": "Consumer Staples",
    "HD": "Consumer Discretionary",
    "CAT": "Industrials",
    "NEM": "Materials",
    "PLD": "Real Estate",
    "DUK": "Utilities",
}

MARKET_PROXY = "SPY"

# Issue #8: richer Treasury key-rate grid (ascending tenor). These DGS* series are continuously
# available from FRED since well before START_DATE (DGS20 is intentionally excluded because it was
# discontinued 1987-1993 and only reintroduced in Oct-2020, which would create a large NaN gap and
# shrink the aligned sample). With 10 tenors the rates PCA genuinely compresses and the
# level/slope/curvature interpretation becomes meaningful rather than forced on 3 points.
RATE_TICKERS = ["DGS1MO", "DGS3MO", "DGS6MO", "DGS1", "DGS2", "DGS3", "DGS5", "DGS7", "DGS10", "DGS30"]
TENOR_YEARS = {
    "DGS1MO": 1.0 / 12.0,
    "DGS3MO": 0.25,
    "DGS6MO": 0.5,
    "DGS1": 1.0,
    "DGS2": 2.0,
    "DGS3": 3.0,
    "DGS5": 5.0,
    "DGS7": 7.0,
    "DGS10": 10.0,
    "DGS30": 30.0,
}

# Sample configuration
START_DATE = "2018-01-02"
AS_OF_DATE = pd.Timestamp("2023-01-01")
DOWNLOAD_END = (AS_OF_DATE + pd.Timedelta(days=10)).strftime("%Y-%m-%d")

# Active equity engine: choose among
#   "latent_statistical"
#   "proxy_market"
#   "apt_ff3"
#   "apt_carhart"
ACTIVE_EQUITY_ENGINE = "apt_carhart"

# Scenario engine configuration
HORIZON_BDAYS = 5
N_SCENARIOS = 10_000
RNG_SEED = 42
VALIDATION_SEED = 2024          # independent seed for the out-of-sample (fresh-draw) check (issue #4)
BOOTSTRAP_EXPECTED_BLOCK = 10
TARGET_RATE_EXPLAINED_VAR = 0.995
TARGET_RESID_EXPLAINED_VAR = 0.95

# Robust portfolio search (min-CVaR is now an exact LP; alpha/max_weight are the live knobs)
CVAR_ALPHA = 0.05
CVAR_MAX_WEIGHT = 0.50

# Walk-forward VaR backtest configuration (issue #2). This is the most expensive cell: it refits
# the engine at each as-of date on live data. Defaults are modest so it completes in reasonable
# time; increase BACKTEST_N_DATES (and re-run) for more statistical power in the coverage tests.
BACKTEST_N_DATES = 40
BACKTEST_STEP_BDAYS = 5
BACKTEST_N_SCENARIOS = 2_000

# Treasury note setup
COUPON_RATE = 0.02
FACE_VALUE = 100.0
PAY_FREQ = 2

REPRESENTATIVE_EQUITY = TICKERS[0]


In [ ]:
# Utilities: data loading, factor preparation, risk-driver extraction, scenario generation, repricing, portfolio analytics

def download_market_data(tickers, market_proxy, rate_tickers, start_date, download_end):
    """Download adjusted-close equity prices, a market proxy, and FRED Treasury key rates."""
    all_equity_tickers = list(dict.fromkeys(list(tickers) + [market_proxy]))
    stock_raw = yf.download(
        all_equity_tickers,
        start=start_date,
        end=download_end,
        auto_adjust=False,
        progress=False,
        actions=False,
    )

    if isinstance(stock_raw.columns, pd.MultiIndex):
        if "Adj Close" not in stock_raw.columns.get_level_values(0):
            raise ValueError("Adjusted close prices not found in yfinance download output.")
        prices_all = stock_raw["Adj Close"].copy()
    else:
        prices_all = stock_raw.copy()
        if isinstance(prices_all, pd.Series):
            prices_all = prices_all.to_frame(name=all_equity_tickers[0])
        prices_all.columns = all_equity_tickers

    stock_prices = prices_all[tickers].copy()
    market_proxy_prices = prices_all[[market_proxy]].copy()
    rates_raw = web.DataReader(rate_tickers, "fred", start_date, download_end).astype(float) / 100.0
    return stock_prices, market_proxy_prices, rates_raw


def _normalize_ff_columns(columns):
    mapping = {}
    for col in columns:
        cleaned = re.sub(r"[^A-Za-z0-9]+", "_", str(col).strip()).upper().strip("_")
        mapping[col] = cleaned
    return mapping


def _ff_table_to_returns(df):
    """Convert raw Fama-French / momentum tables to *simple daily returns* (decimals).

    Issue #7 fix: the FF series are excess / long-short spread returns (MKT-RF, SMB, HML, MOM).
    Applying log1p to a zero-cost long-short spread has no clean interpretation and is
    inconsistent with combining the factor contributions *additively* in the excess-return
    reconstruction. We therefore keep them as simple, additive daily returns. At the daily scale
    log1p(x) ~= x, so this does not move the numbers materially; it only removes the conceptual
    mismatch. (RF is also a level return and is left in the same simple-return convention so that
    'equity excess = equity log-return - RF' uses exactly the same risk-free that MKT-RF embeds.)
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index.astype(str))
    out = out.apply(pd.to_numeric, errors="coerce")
    if out.abs().median().max() > 0.50:
        out = out / 100.0
    return out


def download_fama_french_daily_factors(start_date, end_date):
    """Download daily Fama-French market/style factors plus the daily momentum factor.

    The series are converted to simple daily returns (see `_ff_table_to_returns`) so they can be
    combined additively with the log-return machinery used elsewhere in the notebook.
    """
    ff3 = web.DataReader("F-F_Research_Data_Factors_daily", "famafrench", start=start_date, end=end_date)[0]
    mom = web.DataReader("F-F_Momentum_Factor_daily", "famafrench", start=start_date, end=end_date)[0]

    ff3 = ff3.rename(columns=_normalize_ff_columns(ff3.columns))
    mom = mom.rename(columns=_normalize_ff_columns(mom.columns))

    ff3 = _ff_table_to_returns(ff3)
    mom = _ff_table_to_returns(mom)

    mom_col = None
    for candidate in ["MOM", "MOMENTUM", "MOM_FACTOR"]:
        if candidate in mom.columns:
            mom_col = candidate
            break
    if mom_col is None:
        mom_col = mom.columns[0]

    ff = ff3.join(mom[[mom_col]].rename(columns={mom_col: "MOM"}), how="left")

    keep = [c for c in ["MKT_RF", "SMB", "HML", "RF", "MOM"] if c in ff.columns]
    ff = ff[keep].sort_index().loc[:pd.Timestamp(end_date)]
    return ff.dropna(how="any")


def choose_effective_asof(stock_prices, market_proxy_prices, rates, ff_factors, as_of_date):
    common_dates = stock_prices.index.intersection(market_proxy_prices.index).intersection(rates.index).intersection(ff_factors.index)
    eligible = common_dates[common_dates <= as_of_date]
    if len(eligible) == 0:
        raise ValueError("No common market date available at or before the chosen AS_OF_DATE.")
    return eligible.max()


def align_market_data(stock_prices, market_proxy_prices, rates, ff_factors, effective_date):
    prices = stock_prices.loc[:effective_date].dropna(how="any")
    market = market_proxy_prices.loc[:effective_date].dropna(how="any")
    yields = rates.loc[:effective_date].dropna(how="any")
    ff = ff_factors.loc[:effective_date].dropna(how="any")
    common_dates = prices.index.intersection(market.index).intersection(yields.index).intersection(ff.index)
    return prices.loc[common_dates], market.loc[common_dates], yields.loc[common_dates], ff.loc[common_dates]


def compute_equity_log_returns(prices):
    return np.log(prices).diff().dropna()


def compute_yield_changes(yields):
    return yields.diff().dropna()


def compute_excess_log_returns_daily(log_returns, rf_daily):
    """Excess log returns using a *daily* risk-free series (already per-day, in decimals).

    Issue #6 fix: equity excess returns now use the SAME daily risk-free that defines the
    Fama-French factors (the FF 'RF' column), instead of DGS2/252. This removes the double
    risk-free convention (DGS2 for the equity excess vs the FF RF embedded inside MKT-RF) and
    stops using a 2Y point as an overnight-rate proxy.
    """
    rf = rf_daily.reindex(log_returns.index).ffill()
    excess = log_returns.sub(rf, axis=0)
    return excess, rf


def compute_market_excess_log_returns_daily(market_prices, rf_daily):
    market_log_returns = np.log(market_prices.squeeze()).diff().dropna()
    rf = rf_daily.reindex(market_log_returns.index).ffill()
    market_excess = market_log_returns.sub(rf, axis=0)
    market_excess.name = f"{market_prices.columns[0]}_excess_log_return"
    return market_excess


def plot_time_series(df, title, ylabel=None):
    ax = df.plot(figsize=(11, 6), lw=1.2)
    ax.set_title(title)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_histogram(series, title, xlabel):
    ax = pd.Series(series).plot(kind="hist", bins=60, alpha=0.75)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(True, alpha=0.3)
    plt.show()


def distribution_summary(series):
    s = pd.Series(series, dtype=float).dropna()
    return pd.Series({
        "mean": s.mean(),
        "std": s.std(),
        "q01": s.quantile(0.01),
        "q05": s.quantile(0.05),
        "median": s.median(),
        "q95": s.quantile(0.95),
        "q99": s.quantile(0.99),
        "min": s.min(),
        "max": s.max(),
    })


def loss_summary_from_pnl(series, alpha=0.05):
    s = pd.Series(series, dtype=float).dropna()
    threshold = s.quantile(alpha)
    tail = s[s <= threshold]
    return pd.Series({
        "mean": s.mean(),
        "std": s.std(),
        "q01": s.quantile(0.01),
        "q05": s.quantile(0.05),
        "median": s.median(),
        "q95": s.quantile(0.95),
        "q99": s.quantile(0.99),
        "VaR_95_loss": -threshold,
        "ES_95_loss": -tail.mean() if len(tail) > 0 else np.nan,
    })


def _compress_residual_block(residuals, resid_explained_var_target=0.95):
    resid_pca_full = PCA().fit(residuals)
    resid_cumvar = np.cumsum(resid_pca_full.explained_variance_ratio_)
    k_resid = int(np.searchsorted(resid_cumvar, resid_explained_var_target) + 1)
    k_resid = max(1, min(k_resid, residuals.shape[1] - 1))

    resid_pca = PCA(n_components=k_resid)
    resid_scores = pd.DataFrame(
        resid_pca.fit_transform(residuals),
        index=residuals.index,
        columns=[f"resid_pc{i+1}" for i in range(k_resid)],
    )

    resid_loadings = pd.DataFrame(
        resid_pca.components_.T,
        index=residuals.columns,
        columns=resid_scores.columns,
    )

    resid_explained_ratio = pd.Series(
        resid_pca.explained_variance_ratio_,
        index=resid_scores.columns,
        name="resid_explained_variance_ratio",
    )

    # Issue #11: PCA on the residuals discards the variance beyond the retained components. That
    # dropped idiosyncratic variance never re-enters the simulation, so single-name tails would be
    # understated. We measure, per asset, the standard deviation of the part of the residual NOT
    # captured by the retained PCs, and re-inject it as iid noise in the simulator.
    reconstructed = resid_scores.to_numpy() @ resid_loadings.to_numpy().T
    leftover = residuals.to_numpy() - reconstructed
    resid_idiosyncratic_std = pd.Series(
        leftover.std(axis=0, ddof=1),
        index=residuals.columns,
        name="resid_idiosyncratic_std",
    )

    return {
        "resid_scores": resid_scores,
        "resid_loadings": resid_loadings,
        "resid_explained_ratio": resid_explained_ratio,
        "resid_cum_explained_ratio": resid_explained_ratio.cumsum(),
        "resid_idiosyncratic_std": resid_idiosyncratic_std,
        "resid_pca": resid_pca,
    }


def build_factor_driven_equity_block(excess_returns, factor_df, block_name, resid_explained_var_target=0.95):
    common_dates = excess_returns.index.intersection(factor_df.index)
    y = excess_returns.loc[common_dates].copy()
    f = factor_df.loc[common_dates].copy()

    mu_excess = y.mean()
    mu_factors = f.mean()
    centered_y = y - mu_excess
    centered_f = f - mu_factors

    X = centered_f.to_numpy()
    Y = centered_y.to_numpy()
    beta_array = np.linalg.lstsq(X, Y, rcond=None)[0].T  # assets x factors

    betas = pd.DataFrame(beta_array, index=y.columns, columns=centered_f.columns)
    fitted = pd.DataFrame(X @ beta_array.T, index=y.index, columns=y.columns)
    residuals = centered_y - fitted
    asset_r2 = 1.0 - residuals.var(ddof=1) / centered_y.var(ddof=1)

    residual_block = _compress_residual_block(residuals, resid_explained_var_target)

    return {
        "engine_name": block_name,
        "factors": centered_f,
        "factor_means": mu_factors,
        "betas": betas,
        "residuals": residuals,
        "asset_r2": asset_r2,
        "mean_excess_returns": mu_excess,
        "excess_returns": y,
        **residual_block,
    }


def build_latent_statistical_equity_block(log_returns, rf_daily, resid_explained_var_target=0.95):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    centered = excess_returns - excess_returns.mean()
    pca = PCA(n_components=1)
    factor_scores = pd.DataFrame(
        pca.fit_transform(centered),
        index=centered.index,
        columns=["latent_equity_factor"],
    )
    block = build_factor_driven_equity_block(
        excess_returns=excess_returns,
        factor_df=factor_scores,
        block_name="latent_statistical",
        resid_explained_var_target=resid_explained_var_target,
    )
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["latent_explained_variance"] = float(pca.explained_variance_ratio_[0])
    return block


def build_proxy_market_equity_block(log_returns, market_proxy_prices, rf_daily, resid_explained_var_target=0.95):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    market_excess = compute_market_excess_log_returns_daily(market_proxy_prices, rf_daily)
    factor_df = market_excess.to_frame(name="market_erp_proxy")
    block = build_factor_driven_equity_block(
        excess_returns=excess_returns,
        factor_df=factor_df,
        block_name="proxy_market",
        resid_explained_var_target=resid_explained_var_target,
    )
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["market_excess"] = market_excess.reindex(block["excess_returns"].index)
    return block


def build_apt_equity_block(log_returns, rf_daily, ff_factor_panel, selected_factors, block_name, resid_explained_var_target=0.95):
    excess_returns, short_rate_daily = compute_excess_log_returns_daily(log_returns, rf_daily)
    factor_df = ff_factor_panel[selected_factors].copy()
    block = build_factor_driven_equity_block(
        excess_returns=excess_returns,
        factor_df=factor_df,
        block_name=block_name,
        resid_explained_var_target=resid_explained_var_target,
    )
    block["short_rate_daily"] = short_rate_daily.reindex(block["excess_returns"].index)
    block["selected_factors"] = list(selected_factors)
    return block


def summarize_equity_engine(block):
    return pd.Series({
        "n_equity_factors": block["factors"].shape[1],
        "n_residual_factors": block["resid_scores"].shape[1],
        "mean_asset_R2": block["asset_r2"].mean(),
        "median_asset_R2": block["asset_r2"].median(),
        "max_asset_R2": block["asset_r2"].max(),
        "min_asset_R2": block["asset_r2"].min(),
    }, name=block["engine_name"])


def build_rates_factor_model(yield_changes, explained_var_target=0.995):
    dy_mean = yield_changes.mean()
    dy_centered = yield_changes - dy_mean

    pca_full = PCA().fit(dy_centered)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    k_rates = int(np.searchsorted(cumvar, explained_var_target) + 1)
    k_rates = max(2, min(k_rates, dy_centered.shape[1]))

    pca = PCA(n_components=k_rates)
    scores = pd.DataFrame(
        pca.fit_transform(dy_centered),
        index=dy_centered.index,
        columns=[f"rate_pc{i+1}" for i in range(k_rates)],
    )

    loadings = pd.DataFrame(
        pca.components_.T,
        index=dy_centered.columns,
        columns=scores.columns,
    )

    explained_ratio = pd.Series(
        pca.explained_variance_ratio_,
        index=scores.columns,
        name="explained_variance_ratio",
    )

    return {
        "scores": scores,
        "loadings": loadings,
        "mean_dy": dy_mean,
        "explained_ratio": explained_ratio,
        "cum_explained_ratio": explained_ratio.cumsum(),
        "n_factors": k_rates,
        "pca": pca,
    }


def build_curve_lsc_factors(key_rates):
    """Interpretable level / slope / curvature proxies, generalized to whatever tenors are present.

    Issue #8: with a richer curve we no longer hard-code DGS2/DGS5/DGS10. Assuming the columns are
    in ascending tenor order: level = average yield, slope = longest - shortest,
    curvature = 2*mid - shortest - longest.
    """
    cols = list(key_rates.columns)
    short, long = cols[0], cols[-1]
    mid = cols[len(cols) // 2]
    level = key_rates.mean(axis=1)
    slope = key_rates[long] - key_rates[short]
    curvature = 2.0 * key_rates[mid] - key_rates[short] - key_rates[long]
    return pd.DataFrame({
        "curve_level": level,
        "curve_slope": slope,
        "curve_curvature": curvature,
    }, index=key_rates.index)


def build_joint_driver_matrix(equity_block, rates_block):
    drivers = pd.concat([
        equity_block["factors"],
        equity_block["resid_scores"],
        rates_block["scores"],
    ], axis=1).dropna()
    return drivers


def build_var1_model(drivers):
    return VAR(drivers).fit(maxlags=1, trend="c")


def var_one_step_r2(var_res, drivers):
    """Per-equation in-sample one-step-ahead R^2 of the VAR(1) vs a constant-only baseline.

    Issue #9 diagnostic: if these R^2 are near zero, the VAR(1) mean dynamics add little and the
    simulated risk is effectively coming from the bootstrapped innovations + factor loadings, not
    from genuine cross-predictability. In that case the model could be simplified (drop the VAR and
    block-bootstrap the demeaned joint innovations) or regularized (Minkowski/ridge / block prior).
    """
    resid = var_res.resid
    y = drivers.loc[resid.index]
    r2 = 1.0 - resid.var(ddof=0) / y.var(ddof=0)
    return r2.sort_values(ascending=False)


def stationary_bootstrap_index_matrix(n_obs, n_paths, horizon, avg_block_length, rng):
    """Vectorized stationary bootstrap (Politis-Romano) index matrix, shape (n_paths, horizon).

    Issue #5: replaces the per-path Python loop. Same generative process as before: at each step,
    with probability 1/avg_block_length restart at a fresh uniform index, otherwise advance the
    previous index by one (mod n_obs). Only `horizon` vectorized iterations are needed.
    """
    p_restart = 1.0 / avg_block_length
    restart = rng.random((n_paths, horizon)) < p_restart
    restart[:, 0] = True
    fresh = rng.integers(0, n_obs, size=(n_paths, horizon))
    idx = np.empty((n_paths, horizon), dtype=int)
    idx[:, 0] = fresh[:, 0]
    for h in range(1, horizon):
        advanced = (idx[:, h - 1] + 1) % n_obs
        idx[:, h] = np.where(restart[:, h], fresh[:, h], advanced)
    return idx


def simulate_joint_horizon(var_res, resid_hist, last_state, equity_block, rates_block,
                           current_prices, current_key_rates, horizon_bdays, n_scenarios,
                           avg_block_length, rng_seed, daily_rf, resid_idio_std=None,
                           rate_floor=-0.02, _boot_idx=None):
    """Vectorized one-week joint scenario simulation (issue #5).

    Changes vs the original loop version:
      * fully vectorized across scenarios; only `horizon_bdays` steps are iterated, with batch
        matrix operations over the (n_scenarios, state_dim) state.
      * equity total-return reconstruction adds back a *constant* daily risk-free `daily_rf` (the
        FF daily RF at the as-of date) instead of the simulated DGS2/252, which unifies the
        risk-free convention with the factor block (issue #6). The variation of the short rate over
        a 5-business-day horizon is immaterial (order 1e-5), so treating it as constant is cleaner.
      * optional iid idiosyncratic noise `resid_idio_std` re-injects the residual variance that the
        PCA compression discards, so single-name tails are not understated (issue #11).

    Note: vectorization changes the order in which random numbers are consumed, so individual
    scenario paths differ from the old implementation; the simulated *distribution* is unchanged.
    `_boot_idx` is an internal testing hook (pass a precomputed index matrix to make the run
    deterministic).
    """
    rng = np.random.default_rng(rng_seed)
    intercept = np.asarray(var_res.intercept, dtype=float)            # (k,)
    coef = np.asarray(var_res.coefs[0], dtype=float)                  # (k, k)
    resid_hist_arr = np.asarray(resid_hist, dtype=float)
    n_hist = resid_hist_arr.shape[0]

    asset_names = current_prices.index
    rate_names = current_key_rates.index
    n_stocks = len(asset_names)
    n_eq = equity_block["factors"].shape[1]
    n_resid = equity_block["resid_loadings"].shape[1]
    n_rate = rates_block["loadings"].shape[1]

    mu_excess = equity_block["mean_excess_returns"].to_numpy()        # (n_stocks,)
    beta = equity_block["betas"].to_numpy()                           # (n_stocks, n_eq)
    resid_load = equity_block["resid_loadings"].to_numpy()            # (n_stocks, n_resid)
    mu_dy = rates_block["mean_dy"].to_numpy()                         # (n_key_rates,)
    rate_load = rates_block["loadings"].to_numpy()                    # (n_key_rates, n_rate)

    if resid_idio_std is None:
        idio_std = np.zeros(n_stocks)
    else:
        idio_std = np.nan_to_num(np.asarray(resid_idio_std.reindex(asset_names).to_numpy(), dtype=float), nan=0.0)

    if _boot_idx is None:
        boot_idx = stationary_bootstrap_index_matrix(n_hist, n_scenarios, horizon_bdays, avg_block_length, rng)
    else:
        boot_idx = np.asarray(_boot_idx, dtype=int)

    state = np.tile(np.asarray(last_state, dtype=float), (n_scenarios, 1))                 # (S, k)
    key_rate_path = np.tile(current_key_rates.to_numpy().astype(float), (n_scenarios, 1))  # (S, n_key_rates)
    cum_excess = np.zeros((n_scenarios, n_stocks))
    cum_log = np.zeros((n_scenarios, n_stocks))

    for h in range(horizon_bdays):
        eps = resid_hist_arr[boot_idx[:, h]]                          # (S, k)
        state = intercept[None, :] + state @ coef.T + eps            # (S, k)

        eq_state = state[:, :n_eq]
        resid_state = state[:, n_eq:n_eq + n_resid]
        rate_state = state[:, n_eq + n_resid:n_eq + n_resid + n_rate]

        dy = mu_dy[None, :] + rate_state @ rate_load.T               # (S, n_key_rates)
        key_rate_path = np.clip(key_rate_path + dy, rate_floor, None)

        idio = resid_state @ resid_load.T                            # (S, n_stocks)
        if idio_std.any():
            idio = idio + rng.normal(0.0, 1.0, size=(n_scenarios, n_stocks)) * idio_std[None, :]

        daily_excess = mu_excess[None, :] + eq_state @ beta.T + idio  # (S, n_stocks)
        daily_log = daily_excess + daily_rf                          # constant rf add-back (issue #6)
        cum_excess += daily_excess
        cum_log += daily_log

    price_scenarios = current_prices.to_numpy()[None, :] * np.exp(cum_log)

    return {
        "stock_excess_return_scenarios": pd.DataFrame(cum_excess, columns=asset_names),
        "stock_log_return_scenarios": pd.DataFrame(cum_log, columns=asset_names),
        "stock_price_scenarios": pd.DataFrame(price_scenarios, columns=asset_names),
        "key_rate_scenarios": pd.DataFrame(key_rate_path, columns=rate_names),
    }


def make_zero_curve_from_key_rates(key_tenors_years, key_rates):
    """Build a linear-interpolated zero curve from tenor/rate pairs.

    Robustness note: both tenors and rates are converted to numpy arrays and sorted by tenor. This
    avoids relying on pandas Series positional indexing semantics and keeps the curve construction
    stable if the caller passes a Series rather than a plain array.
    """
    tenors = np.asarray(key_tenors_years, dtype=float)
    rates = np.asarray(key_rates, dtype=float)
    order = np.argsort(tenors)
    tenors = tenors[order]
    rates = rates[order]
    x = np.concatenate([[0.0], tenors])
    y = np.concatenate([[rates[0]], rates])
    def zero_rate(t):
        t = float(max(t, 0.0))
        return float(np.interp(t, x, y))
    return zero_rate


def generate_coupon_schedule(issue_date, maturity_date, freq=2):
    months_step = 12 // freq
    dates = []
    current = pd.Timestamp(issue_date) + pd.DateOffset(months=months_step)
    while current < pd.Timestamp(maturity_date):
        dates.append(current)
        current = current + pd.DateOffset(months=months_step)
    dates.append(pd.Timestamp(maturity_date))
    return pd.DatetimeIndex(dates)


def price_coupon_bond_from_curve(eval_date, payment_dates, key_rate_series, tenor_years,
                                 coupon_rate=0.02, face_value=100.0, freq=2):
    """Dirty present value of a coupon bond off an interpolated zero curve.

    Issue #8: the tenor grid is now passed in (`tenor_years`) instead of hard-coding [2, 5, 10],
    so the pricer works with any set of FRED key rates.

    Issue #10 (declared approximations): the FRED CMT series (DGS*) are *par* bond-equivalent
    yields, not zero rates. Here we (a) linearly interpolate them across tenor and (b) treat the
    interpolated yields as *continuously-compounded zero rates* (exp(-z t) discounting). This
    conflates par-yield ~= zero-rate and bond-equivalent ~= continuous compounding and introduces a
    small systematic bias. It is acceptable for a prototype; a production version would bootstrap a
    zero curve from the par yields and respect the day-count / compounding convention.

    Returns the *dirty* (full) present value of all remaining cashflows; see `accrued_interest`
    for the clean/dirty split.
    """
    remaining_dates = payment_dates[payment_dates > pd.Timestamp(eval_date)]
    if len(remaining_dates) == 0:
        return float(face_value)
    times = np.array([(d - pd.Timestamp(eval_date)).days / 365.25 for d in remaining_dates], dtype=float)
    zero_curve = make_zero_curve_from_key_rates(tenor_years, np.asarray(key_rate_series, dtype=float))
    zero_rates = np.array([zero_curve(t) for t in times], dtype=float)
    discount_factors = np.exp(-zero_rates * times)
    cashflows = np.full(len(times), coupon_rate * face_value / freq, dtype=float)
    cashflows[-1] += face_value
    return float(np.sum(cashflows * discount_factors))


def accrued_interest(eval_date, payment_dates, coupon_rate=0.02, face_value=100.0, freq=2):
    """Accrued interest at eval_date (actual/actual within the current coupon period).

    Issue #10: included so portfolio P&L can be reported on a carry-consistent (clean vs dirty)
    basis, with clean = dirty - accrued. Over a one-week horizon with semiannual coupons this is
    immaterial (order 0.03 on a 2% note) but it is the technically-correct decomposition.
    """
    eval_date = pd.Timestamp(eval_date)
    future = payment_dates[payment_dates > eval_date]
    if len(future) == 0:
        return 0.0
    next_cpn = future[0]
    months_step = 12 // freq
    prev_cpn = next_cpn - pd.DateOffset(months=months_step)
    period_days = (next_cpn - prev_cpn).days
    accrued_days = max((eval_date - prev_cpn).days, 0)
    frac = accrued_days / period_days if period_days > 0 else 0.0
    return float(coupon_rate * face_value / freq * frac)


def rolling_horizon_log_returns(log_returns, window):
    return log_returns.rolling(window).sum().dropna()


def rolling_horizon_yield_changes(yield_changes, window):
    return yield_changes.rolling(window).sum().dropna()


def portfolio_metrics_from_instrument_returns(weights, instrument_returns, alpha=0.05):
    w = pd.Series(weights, index=instrument_returns.columns, dtype=float)
    w = w / w.sum()
    portfolio_return = instrument_returns.mul(w, axis=1).sum(axis=1)
    metrics = loss_summary_from_pnl(portfolio_return, alpha=alpha)
    metrics["mean_abs_weight"] = w.abs().mean()
    return portfolio_return, metrics


def inverse_vol_weights_from_scenarios(instrument_returns):
    vol = instrument_returns.std()
    inv_vol = 1.0 / vol.replace(0, np.nan)
    w = inv_vol / inv_vol.sum()
    return w.fillna(0.0)


def tail_contribution_table(portfolio_return, instrument_returns, weights, alpha=0.05):
    w = pd.Series(weights, index=instrument_returns.columns, dtype=float)
    w = w / w.sum()
    threshold = portfolio_return.quantile(alpha)
    tail_mask = portfolio_return <= threshold
    if tail_mask.sum() == 0:
        return pd.DataFrame()

    weighted_return_contrib = instrument_returns.mul(w, axis=1)
    avg_tail_return_contrib = weighted_return_contrib.loc[tail_mask].mean()
    avg_tail_loss_contrib = -avg_tail_return_contrib
    portfolio_es_loss = float(avg_tail_loss_contrib.sum())
    share = avg_tail_loss_contrib / portfolio_es_loss if portfolio_es_loss != 0 else np.nan

    return pd.DataFrame({
        "weight": w,
        "avg_tail_return_contribution": avg_tail_return_contrib,
        "avg_tail_loss_contribution": avg_tail_loss_contrib,
        "share_of_portfolio_ES_loss": share,
    }).sort_values("avg_tail_loss_contribution", ascending=False)


def search_long_only_min_cvar_portfolio(instrument_returns, alpha=0.05, max_weight=None,
                                        n_draws=None, seed=None, chunk_size=None):
    """Exact long-only minimum-CVaR portfolio via the Rockafellar-Uryasev LP (issue #3).

    Replaces the previous random simplex search with the convex linear program

        min_{w, zeta, u}   zeta + 1/(alpha * S) * sum_s u_s
        s.t.   u_s >= -(r_s . w) - zeta,   u_s >= 0
               sum_i w_i = 1,   0 <= w_i <= max_weight

    where r_s is the instrument-return vector in scenario s and S the number of scenarios. Solved
    with scipy.optimize.linprog(method='highs'). This returns the true optimum, is deterministic
    and reproducible, scales with the number of assets, and does not discard most draws under a
    tight max-weight cap (the old rejection sampler did). The legacy `n_draws / seed / chunk_size`
    arguments are accepted but ignored for backward compatibility.
    """
    R = instrument_returns.to_numpy()
    S, n = R.shape
    names = instrument_returns.columns.tolist()

    c = np.concatenate([np.zeros(n), [1.0], np.full(S, 1.0 / (alpha * S))])

    # inequality block:  -(R w) - zeta - u <= 0   (one row per scenario)
    A_ub = sparse.hstack([
        sparse.csr_matrix(-R),
        sparse.csr_matrix(-np.ones((S, 1))),
        -sparse.identity(S, format="csr"),
    ], format="csr")
    b_ub = np.zeros(S)

    # equality block:  sum w = 1
    A_eq = sparse.hstack([
        sparse.csr_matrix(np.ones((1, n))),
        sparse.csr_matrix((1, 1)),
        sparse.csr_matrix((1, S)),
    ], format="csr")
    b_eq = np.array([1.0])

    w_upper = max_weight if max_weight is not None else 1.0
    bounds = [(0.0, w_upper)] * n + [(None, None)] + [(0.0, None)] * S

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    if not res.success:
        raise RuntimeError(f"min-CVaR LP failed to solve: {res.message}")

    w = np.clip(res.x[:n], 0.0, None)
    w = w / w.sum()
    port = R @ w
    q = float(np.quantile(port, alpha))
    tail = port[port <= q]
    es = float(-tail.mean()) if tail.size else np.nan

    weights = pd.Series(w, index=names, name="min_cvar_long_only")
    diagnostics = pd.Series({
        "n_scenarios": float(S),
        "VaR_95_loss": float(-q),
        "ES_95_loss": es,
        "mean": float(port.mean()),
        "std": float(port.std(ddof=1)),
    }, name="min_cvar_lp_diagnostics")
    return weights, diagnostics


def kupiec_pof_test(n_violations, n_obs, alpha):
    """Kupiec proportion-of-failures (unconditional coverage) LR test.

    H0: the VaR violation rate equals `alpha`. Returns (LR_stat, p_value); LR_stat ~ chi2(1).
    The expected number of violations is alpha * n_obs.
    """
    x, N, p = int(n_violations), int(n_obs), float(alpha)
    if N == 0:
        return np.nan, np.nan
    pi_hat = x / N
    if x == 0:
        lr = -2.0 * (N * np.log(1 - p))
    elif x == N:
        lr = -2.0 * (N * np.log(p))
    else:
        ll_null = x * np.log(p) + (N - x) * np.log(1 - p)
        ll_alt = x * np.log(pi_hat) + (N - x) * np.log(1 - pi_hat)
        lr = -2.0 * (ll_null - ll_alt)
    return float(lr), float(chi2.sf(lr, df=1))


def christoffersen_independence_test(violations):
    """Christoffersen independence LR test on a 0/1 violation sequence.

    H0: violations are serially independent (no clustering of breaches). Returns (LR_stat, p_value);
    LR_stat ~ chi2(1).
    """
    v = np.asarray(violations, dtype=int)
    if len(v) < 2:
        return np.nan, np.nan
    n00 = n01 = n10 = n11 = 0
    for prev, cur in zip(v[:-1], v[1:]):
        if prev == 0 and cur == 0:
            n00 += 1
        elif prev == 0 and cur == 1:
            n01 += 1
        elif prev == 1 and cur == 0:
            n10 += 1
        else:
            n11 += 1
    denom01 = n00 + n01
    denom11 = n10 + n11
    pi01 = n01 / denom01 if denom01 > 0 else 0.0
    pi11 = n11 / denom11 if denom11 > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    def _safe(p):
        return min(max(p, 1e-12), 1 - 1e-12)

    ll_null = (n00 + n10) * np.log(1 - _safe(pi)) + (n01 + n11) * np.log(_safe(pi))
    ll_alt = (n00 * np.log(1 - _safe(pi01)) + n01 * np.log(_safe(pi01))
              + n10 * np.log(1 - _safe(pi11)) + n11 * np.log(_safe(pi11)))
    lr = -2.0 * (ll_null - ll_alt)
    return float(lr), float(chi2.sf(lr, df=1))



## 1. Load and align market data

The equity universe is now intentionally broader than the earlier tech-only prototype. The cross-asset input layer includes:

- a **multi-sector US equity sample**,
- a **market proxy**,
- **Treasury key rates**,
- and daily **observable factor data** used by the APT engine.


In [ ]:

# Download and align the data
stock_prices_raw, market_proxy_raw, rates_raw = download_market_data(
    tickers=TICKERS,
    market_proxy=MARKET_PROXY,
    rate_tickers=RATE_TICKERS,
    start_date=START_DATE,
    download_end=DOWNLOAD_END,
)

ff_factor_panel_raw = download_fama_french_daily_factors(
    start_date=START_DATE,
    end_date=DOWNLOAD_END,
)

effective_date = choose_effective_asof(
    stock_prices_raw,
    market_proxy_raw,
    rates_raw,
    ff_factor_panel_raw,
    AS_OF_DATE,
)

stock_prices, market_proxy_prices, key_rates, ff_factor_panel = align_market_data(
    stock_prices_raw,
    market_proxy_raw,
    rates_raw,
    ff_factor_panel_raw,
    effective_date,
)

sector_table = pd.Series(SECTOR_MAP, name="sector").to_frame()

print(f"Configured AS_OF_DATE      : {AS_OF_DATE.date()}")
print(f"Effective evaluation date : {effective_date.date()}")
print(f"Equity sample shape       : {stock_prices.shape}")
print(f"Market proxy sample shape : {market_proxy_prices.shape}")
print(f"Rates sample shape        : {key_rates.shape}")
print(f"FF factor sample shape    : {ff_factor_panel.shape}")
print(f"Active ERP engine         : {ACTIVE_EQUITY_ENGINE}")

print()
print("Equity universe and sectors:")
display(sector_table)

print("Latest cross-asset state at the effective date:")
display(pd.concat([stock_prices.tail(1).T, market_proxy_prices.tail(1).T, key_rates.tail(1).T], axis=0))


In [ ]:
# Visual inspection of raw state variables
plot_time_series(stock_prices, "Adjusted-close prices — broad US equity prototype universe", "Price")
plot_time_series(market_proxy_prices, f"Adjusted-close price — market proxy ({MARKET_PROXY})", "Price")
plot_time_series(key_rates * 100.0, "Treasury key rates — full key-rate grid (%)", "Yield (%)")
plot_time_series(ff_factor_panel[[c for c in ["MKT_RF", "SMB", "HML", "MOM"] if c in ff_factor_panel.columns]],
                 "Daily observable equity factor panel (log-return approximations)",
                 "Daily factor return")



## 2. Risk-driver design and modular ERP engines

The equity block is now **plug-and-play**. The notebook builds several alternative engines and lets the user choose the active one:

- **latent_statistical**: latent common factor extracted from equity excess returns,
- **proxy_market**: observed market ERP proxy via the broad market index,
- **apt_ff3**: observable factor engine based on MKT-RF, SMB, HML,
- **apt_carhart**: observable factor engine based on MKT-RF, SMB, HML, MOM.

Each engine is paired with a compressed idiosyncratic residual block.


In [ ]:
# Build transformed risk drivers and alternative equity-premium engines
equity_log_returns = compute_equity_log_returns(stock_prices)
yield_changes = compute_yield_changes(key_rates)
market_log_returns = compute_equity_log_returns(market_proxy_prices)

common_dates = equity_log_returns.index.intersection(yield_changes.index).intersection(market_log_returns.index).intersection(ff_factor_panel.index)
equity_log_returns = equity_log_returns.loc[common_dates]
yield_changes = yield_changes.loc[common_dates]
key_rates = key_rates.loc[common_dates]
market_proxy_prices = market_proxy_prices.loc[common_dates]
ff_factor_panel = ff_factor_panel.loc[common_dates]

# Issue #6: single risk-free convention. The daily Fama-French RF (already downloaded) is the same
# risk-free that MKT_RF embeds, so we use it to define equity excess returns instead of DGS2/252.
rf_daily = ff_factor_panel["RF"]

latent_equity_block = build_latent_statistical_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

proxy_equity_block = build_proxy_market_equity_block(
    log_returns=equity_log_returns,
    market_proxy_prices=market_proxy_prices,
    rf_daily=rf_daily,
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

apt_ff3_equity_block = build_apt_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    ff_factor_panel=ff_factor_panel,
    selected_factors=["MKT_RF", "SMB", "HML"],
    block_name="apt_ff3",
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

apt_carhart_equity_block = build_apt_equity_block(
    log_returns=equity_log_returns,
    rf_daily=rf_daily,
    ff_factor_panel=ff_factor_panel,
    selected_factors=["MKT_RF", "SMB", "HML", "MOM"],
    block_name="apt_carhart",
    resid_explained_var_target=TARGET_RESID_EXPLAINED_VAR,
)

equity_engines = {
    "latent_statistical": latent_equity_block,
    "proxy_market": proxy_equity_block,
    "apt_ff3": apt_ff3_equity_block,
    "apt_carhart": apt_carhart_equity_block,
}

if ACTIVE_EQUITY_ENGINE not in equity_engines:
    raise ValueError(f"Unknown ACTIVE_EQUITY_ENGINE: {ACTIVE_EQUITY_ENGINE}")

equity_block = equity_engines[ACTIVE_EQUITY_ENGINE]

rates_block = build_rates_factor_model(
    yield_changes=yield_changes,
    explained_var_target=TARGET_RATE_EXPLAINED_VAR,
)

curve_lsc_levels = build_curve_lsc_factors(key_rates)
curve_lsc_changes = curve_lsc_levels.diff().dropna()

engine_summary = pd.concat([
    summarize_equity_engine(block) for block in equity_engines.values()
], axis=1).T.sort_values(["mean_asset_R2", "median_asset_R2"], ascending=False)

print(f"Active equity factor block: {equity_block['engine_name']}")
print(f"Rates factors retained   : {rates_block['n_factors']} (from {key_rates.shape[1]} tenors)")
print(f"Daily risk-free source   : Fama-French RF (mean {rf_daily.mean():.6f}/day)")
print("Equity engine comparison:")
display(engine_summary)


In [ ]:

# Diagnostics on engine structure and factor interpretation
active_beta_table = equity_block["betas"].copy()
active_beta_table.index.name = "asset"

rate_interpretation = pd.concat([rates_block["scores"], curve_lsc_changes], axis=1).dropna().corr()
rate_interpretation = rate_interpretation.loc[rates_block["scores"].columns, curve_lsc_changes.columns]

compare_factor_table = pd.concat([
    proxy_equity_block["factors"].rename(columns={proxy_equity_block["factors"].columns[0]: "market_erp_proxy"}),
    latent_equity_block["factors"].rename(columns={latent_equity_block["factors"].columns[0]: "latent_equity_factor"}),
    ff_factor_panel[[c for c in ["MKT_RF", "MOM"] if c in ff_factor_panel.columns]],
], axis=1).dropna().corr()

print("Active engine beta matrix:")
display(active_beta_table)

print("Active engine asset-level R^2:")
display(equity_block["asset_r2"].to_frame("asset_R2"))

print("Residual equity PCA explained variance (active engine):")
display(equity_block["resid_explained_ratio"].to_frame("resid_explained_variance_ratio"))
display(equity_block["resid_cum_explained_ratio"].to_frame("resid_cum_explained_ratio"))

print("Comparison across observable and latent equity factors:")
display(compare_factor_table)

print("Economic interpretation of rates PCA scores via level / slope / curvature changes:")
display(rate_interpretation)

plot_time_series(
    pd.concat([equity_block["factors"], rates_block["scores"]], axis=1),
    f"Active ERP engine factors and rates PCA scores — {ACTIVE_EQUITY_ENGINE}",
    "Factor value",
)



## 3. Joint state representation and hybrid scenario engine

The active joint state vector stacks:

- the chosen **ERP engine factors**,
- compressed **equity residual factors**,
- compressed **risk-free curve factors**.

Dynamics are modelled through a **VAR(1)**, while innovations are generated with **stationary bootstrap** to avoid relying purely on Gaussian i.i.d. shocks.


In [ ]:
# Joint state vector and dynamic model
drivers = build_joint_driver_matrix(equity_block, rates_block)
print(f"Joint driver matrix shape: {drivers.shape}")
print("Driver columns:")
print(drivers.columns.tolist())
display(drivers.head())

var_res = build_var1_model(drivers)
print("VAR(1) fitted successfully.")
print(f"Selected lag order: {var_res.k_ar}")
print(f"Number of state variables: {var_res.neqs}")
display(var_res.params)

try:
    print(var_res.summary())
except Exception as err:
    print("VAR summary could not be rendered cleanly.")
    print(f"Reason: {type(err).__name__}: {err}")

# Issue #9: stability check (statsmodels convention: stable iff ALL companion roots have modulus > 1)
print("Companion roots (modulus should be > 1 for stability):")
print(np.abs(var_res.roots))
print(f"VAR(1) is_stable(): {var_res.is_stable()}")
print(f"Min |root|: {np.min(np.abs(var_res.roots)):.4f}")

# Issue #9: is the VAR(1) doing real work, or is risk coming from the bootstrapped innovations?
# Per-equation one-step in-sample R^2 vs a constant-only baseline.
one_step_r2 = var_one_step_r2(var_res, drivers)
print()
print("VAR(1) one-step-ahead in-sample R^2 per driver:")
display(one_step_r2.to_frame("one_step_R2"))
print(
    f"Mean one-step R^2 across drivers: {one_step_r2.mean():.4f}. "
    "If these are near zero, the VAR mean dynamics add little predictive content and the simulated "
    "risk is effectively driven by the bootstrapped joint innovations + factor loadings; the model "
    "could then be simplified (block-bootstrap the demeaned innovations) or regularized."
)


In [ ]:
# Simulate one-week joint scenarios under the active ERP engine
current_prices = stock_prices.loc[effective_date]
current_key_rates = key_rates.loc[effective_date]

# Issue #6: constant daily risk-free for the excess->total reconstruction, taken as the latest
# Fama-French RF (same convention used to define the excess returns). Issue #11: re-inject the
# idiosyncratic residual variance dropped by the PCA compression as iid noise.
current_daily_rf = float(ff_factor_panel["RF"].iloc[-1])

sim = simulate_joint_horizon(
    var_res=var_res,
    resid_hist=var_res.resid,
    last_state=drivers.iloc[-1].to_numpy(),
    equity_block=equity_block,
    rates_block=rates_block,
    current_prices=current_prices,
    current_key_rates=current_key_rates,
    horizon_bdays=HORIZON_BDAYS,
    n_scenarios=N_SCENARIOS,
    avg_block_length=BOOTSTRAP_EXPECTED_BLOCK,
    rng_seed=RNG_SEED,
    daily_rf=current_daily_rf,
    resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
)

stock_excess_return_scenarios = sim["stock_excess_return_scenarios"]
stock_log_return_scenarios = sim["stock_log_return_scenarios"]
stock_price_scenarios = sim["stock_price_scenarios"]
key_rate_scenarios = sim["key_rate_scenarios"]


In [ ]:

# Marginal forecast views
plot_histogram(
    stock_excess_return_scenarios[REPRESENTATIVE_EQUITY],
    f"Forecast distribution — {REPRESENTATIVE_EQUITY} one-week excess log-return",
    "One-week excess log-return",
)

plot_histogram(
    key_rate_scenarios["DGS10"] * 100.0,
    "Forecast distribution — 10Y Treasury yield at horizon",
    "Yield (%)",
)

display(loss_summary_from_pnl(stock_excess_return_scenarios[REPRESENTATIVE_EQUITY]).to_frame(f"{REPRESENTATIVE_EQUITY}_1w_excess_return"))
display(distribution_summary(key_rate_scenarios["DGS10"]).to_frame("DGS10_horizon_yield"))


In [ ]:
# Distributional and dependence validation (issues #1a, #1c)
# A joint scenario engine must be judged on its dependence and tail calibration, not only on a
# single-name standard deviation. We compare simulated one-week behaviour with historical 5-day
# behaviour along four dimensions; the VaR coverage / PIT calibration is handled separately by the
# walk-forward backtest below.

hist_5d_stock = rolling_horizon_log_returns(equity_log_returns, HORIZON_BDAYS)
hist_5d_yield = rolling_horizon_yield_changes(yield_changes, HORIZON_BDAYS)
sim_d10 = key_rate_scenarios["DGS10"] - current_key_rates["DGS10"]

# (1) Risk scaling: simulated one-week std vs historical rolling 5-day std
scaling_table = pd.DataFrame({
    "hist_5d_std": [hist_5d_stock[REPRESENTATIVE_EQUITY].std(), hist_5d_yield["DGS10"].std()],
    "sim_1w_std": [stock_log_return_scenarios[REPRESENTATIVE_EQUITY].std(), sim_d10.std()],
}, index=[f"{REPRESENTATIVE_EQUITY}_return", "DGS10_change"])
scaling_table["sim/hist"] = scaling_table["sim_1w_std"] / scaling_table["hist_5d_std"]
print("Risk scaling (simulated one-week vs historical 5-day std):")
display(scaling_table)

# (2) Tail behaviour: excess kurtosis (pandas .kurt() is the Fisher / excess kurtosis)
def _exkurt(x):
    return float(pd.Series(x).kurt())

kurt_table = pd.DataFrame({
    "hist_5d_excess_kurtosis": [_exkurt(hist_5d_stock[REPRESENTATIVE_EQUITY]), _exkurt(hist_5d_yield["DGS10"])],
    "sim_1w_excess_kurtosis": [_exkurt(stock_log_return_scenarios[REPRESENTATIVE_EQUITY]), _exkurt(sim_d10)],
}, index=[f"{REPRESENTATIVE_EQUITY}_return", "DGS10_change"])
print("Tail behaviour (excess kurtosis, simulated vs historical 5-day):")
display(kurt_table)

# (3) Cross-equity dependence: simulated vs historical 5-day correlation matrices
sim_corr = stock_log_return_scenarios.corr()
hist_corr = hist_5d_stock.corr()

def _avg_offdiag(corr_df):
    arr = corr_df.to_numpy().copy()
    np.fill_diagonal(arr, np.nan)
    return float(np.nanmean(arr))

cross_equity_compare = pd.Series({
    "sim_avg_cross_equity_corr": _avg_offdiag(sim_corr),
    "hist_avg_cross_equity_corr": _avg_offdiag(hist_corr),
    "max_abs_pairwise_corr_diff": float(np.nanmax(np.abs(sim_corr.to_numpy() - hist_corr.to_numpy()))),
})
print("Cross-equity dependence (simulated vs historical 5-day):")
display(cross_equity_compare.to_frame("value"))

# (4) Equity-vs-rates dependence: corr(equity 5-day return, 5-day change in DGS10)
sim_eq_rate_corr = pd.Series(
    {t: float(np.corrcoef(stock_log_return_scenarios[t], sim_d10)[0, 1]) for t in TICKERS}
)
hist_joint = pd.concat([hist_5d_stock, hist_5d_yield["DGS10"].rename("DGS10_chg")], axis=1).dropna()
hist_eq_rate_corr = pd.Series({t: float(hist_joint[t].corr(hist_joint["DGS10_chg"])) for t in TICKERS})
eq_rate_compare = pd.DataFrame({"sim": sim_eq_rate_corr, "hist": hist_eq_rate_corr})
eq_rate_compare.loc["AVERAGE"] = eq_rate_compare.mean()
print("Equity vs DGS10-change correlation (simulated vs historical 5-day):")
display(eq_rate_compare)

# Marginal distribution shape, historical vs simulated
plot_histogram(
    hist_5d_stock[REPRESENTATIVE_EQUITY],
    f"Historical rolling 5-day {REPRESENTATIVE_EQUITY} log-returns",
    "5-day log-return",
)
plot_histogram(
    stock_log_return_scenarios[REPRESENTATIVE_EQUITY],
    f"Simulated one-week {REPRESENTATIVE_EQUITY} log-returns",
    "one-week log-return",
)


### 3b. Walk-forward VaR backtest (coverage and calibration)

A single as-of date cannot validate a scenario engine. Here the engine is **refit on an expanding window** at a panel of past as-of dates; at each date we simulate the one-week distribution of an **equal-weight equity** portfolio, take its 95% VaR, and compare it with the **realized** forward one-week return. We then run the **Kupiec proportion-of-failures** test (unconditional coverage) and the **Christoffersen independence** test (no clustering of breaches), and inspect the **PIT / rank histogram** (which should be roughly uniform if the predictive distribution is well calibrated).

This is the most expensive cell because it refits the full pipeline at each date on live data; the defaults (`BACKTEST_N_DATES`, `BACKTEST_N_SCENARIOS`) are deliberately modest. Increase them for more statistical power.


In [ ]:
# Walk-forward VaR backtest with Kupiec (POF) and Christoffersen (independence) tests, plus a PIT
# histogram (issues #1b, #1d, #2). The engine is refit on an expanding window at each as-of date.

def _build_active_equity_block(log_returns, market_prices, ff_panel, rf_d):
    if ACTIVE_EQUITY_ENGINE == "latent_statistical":
        return build_latent_statistical_equity_block(
            log_returns, rf_d, TARGET_RESID_EXPLAINED_VAR
        )
    if ACTIVE_EQUITY_ENGINE == "proxy_market":
        return build_proxy_market_equity_block(
            log_returns, market_prices, rf_d, TARGET_RESID_EXPLAINED_VAR
        )
    if ACTIVE_EQUITY_ENGINE == "apt_ff3":
        return build_apt_equity_block(
            log_returns,
            rf_d,
            ff_panel,
            ["MKT_RF", "SMB", "HML"],
            "apt_ff3",
            TARGET_RESID_EXPLAINED_VAR,
        )
    if ACTIVE_EQUITY_ENGINE == "apt_carhart":
        return build_apt_equity_block(
            log_returns,
            rf_d,
            ff_panel,
            ["MKT_RF", "SMB", "HML", "MOM"],
            "apt_carhart",
            TARGET_RESID_EXPLAINED_VAR,
        )
    raise ValueError(f"Unknown ACTIVE_EQUITY_ENGINE: {ACTIVE_EQUITY_ENGINE}")


def _fit_engine_for_asof(asof):
    eff = choose_effective_asof(
        stock_prices_raw,
        market_proxy_raw,
        rates_raw,
        ff_factor_panel_raw,
        asof,
    )

    sp, mp, kr, ff = align_market_data(
        stock_prices_raw,
        market_proxy_raw,
        rates_raw,
        ff_factor_panel_raw,
        eff,
    )

    lr = compute_equity_log_returns(sp)
    dy = compute_yield_changes(kr)
    mlr = compute_equity_log_returns(mp)

    cd = lr.index.intersection(dy.index).intersection(mlr.index).intersection(ff.index)

    lr = lr.loc[cd]
    dy = dy.loc[cd]
    kr = kr.loc[cd]
    mp = mp.loc[cd]
    ff = ff.loc[cd]

    rf_d = ff["RF"]

    eb = _build_active_equity_block(lr, mp, ff, rf_d)
    rb = build_rates_factor_model(dy, TARGET_RATE_EXPLAINED_VAR)
    drv = build_joint_driver_matrix(eb, rb)
    vr = build_var1_model(drv)

    return eff, sp, kr, eb, rb, drv, vr, float(rf_d.iloc[-1])


# Choose the panel of as-of dates from the trading calendar, ensuring a forward window exists.
_cal = pd.DatetimeIndex(stock_prices_raw.index)
_asof_limit = pd.Timestamp(AS_OF_DATE)

# Robust boolean mask: works whether the comparison returns ndarray, Index, or Series-like object.
_mask = np.asarray(_cal <= _asof_limit, dtype=bool)

_elig = np.flatnonzero(_mask)
_elig = _elig[_elig + HORIZON_BDAYS < len(_cal)]

_sel_positions = sorted(
    list(_elig[::-1][::BACKTEST_STEP_BDAYS][:BACKTEST_N_DATES])
)

backtest_asof_dates = [_cal[p] for p in _sel_positions]

if len(backtest_asof_dates) == 0:
    raise ValueError(
        "No eligible backtest as-of dates found. Check AS_OF_DATE, HORIZON_BDAYS, "
        "BACKTEST_STEP_BDAYS, and the available stock_prices_raw calendar."
    )

print(
    f"Walk-forward backtest: {len(backtest_asof_dates)} as-of dates, "
    f"{BACKTEST_N_SCENARIOS} scenarios each, equal-weight equity portfolio, 95% VaR."
)
print(
    f"From {backtest_asof_dates[0].date()} to {backtest_asof_dates[-1].date()} "
    f"(step {BACKTEST_STEP_BDAYS} business days). This refits the engine at each date.\n"
)

bt_records = []

for j, asof in enumerate(backtest_asof_dates):
    try:
        eff, sp, kr, eb, rb, drv, vr, rf_last = _fit_engine_for_asof(asof)

        sim_bt = simulate_joint_horizon(
            var_res=vr,
            resid_hist=vr.resid,
            last_state=drv.iloc[-1].to_numpy(),
            equity_block=eb,
            rates_block=rb,
            current_prices=sp.loc[eff],
            current_key_rates=kr.loc[eff],
            horizon_bdays=HORIZON_BDAYS,
            n_scenarios=BACKTEST_N_SCENARIOS,
            avg_block_length=BOOTSTRAP_EXPECTED_BLOCK,
            rng_seed=RNG_SEED + j,
            daily_rf=rf_last,
            resid_idio_std=eb.get("resid_idiosyncratic_std"),
        )

        # Equal-weight equity portfolio one-week log return:
        # mean of single-name cumulative log returns across selected tickers.
        port_sim = sim_bt["stock_log_return_scenarios"][TICKERS].mean(axis=1)
        var95 = float(-port_sim.quantile(0.05))

        # Locate the effective as-of date in the original trading calendar.
        eff_pos_arr = stock_prices_raw.index.get_indexer([eff])
        if len(eff_pos_arr) == 0 or eff_pos_arr[0] < 0:
            print(f"  [skip {pd.Timestamp(asof).date()}] effective date not found in stock calendar")
            continue

        eff_pos = int(eff_pos_arr[0])
        fwd_pos = eff_pos + HORIZON_BDAYS

        if fwd_pos >= len(stock_prices_raw.index):
            continue

        p0 = stock_prices_raw[TICKERS].iloc[eff_pos]
        p1 = stock_prices_raw[TICKERS].iloc[fwd_pos]

        if p0.isna().any() or p1.isna().any():
            print(f"  [skip {pd.Timestamp(asof).date()}] missing realized forward prices")
            continue

        realized = float(np.log(p1 / p0).mean())

        bt_records.append(
            {
                "asof": eff,
                "VaR_95": var95,
                "realized_1w_return": realized,
                "violation": int(realized < -var95),
                "pit": float((port_sim <= realized).mean()),
            }
        )

    except Exception as err:
        print(f"  [skip {pd.Timestamp(asof).date()}] {type(err).__name__}: {err}")
        continue

    if (j + 1) % 10 == 0:
        print(f"  ...processed {j + 1}/{len(backtest_asof_dates)} as-of dates")

bt = pd.DataFrame(bt_records)

print(f"\nUsable backtest observations: {len(bt)}")

if len(bt) >= 10:
    n_obs = len(bt)
    n_viol = int(bt["violation"].sum())
    expected = 0.05 * n_obs

    lr_k, p_k = kupiec_pof_test(n_viol, n_obs, 0.05)
    lr_c, p_c = christoffersen_independence_test(bt["violation"].to_numpy())

    summary = pd.Series(
        {
            "observations": n_obs,
            "violations": n_viol,
            "expected_violations_at_5pct": expected,
            "empirical_violation_rate": n_viol / n_obs,
            "kupiec_POF_LR": lr_k,
            "kupiec_POF_pvalue": p_k,
            "christoffersen_indep_LR": lr_c,
            "christoffersen_indep_pvalue": p_c,
        },
        name="walk_forward_VaR_backtest",
    )

    print("\nVaR backtest (95%, equal-weight equity, one-week horizon):")
    display(summary.to_frame("value"))

    print(
        "Kupiec p-value tests unconditional coverage (H0: true violation rate = 5%); "
        "Christoffersen p-value tests independence (H0: no clustering of breaches). "
        "p-values above 0.05 do not reject a well-calibrated VaR."
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].hist(bt["pit"], bins=10, range=(0, 1), edgecolor="white", alpha=0.85)
    axes[0].axhline(len(bt) / 10.0, color="black", lw=1, ls="--", label="uniform")
    axes[0].set_title("PIT / rank histogram (should be ~uniform)")
    axes[0].set_xlabel("PIT of realized return")
    axes[0].legend()

    axes[1].plot(bt["asof"], bt["VaR_95"], label="VaR 95% (loss)", lw=1.3)
    axes[1].plot(
        bt["asof"],
        -bt["realized_1w_return"],
        label="realized loss",
        lw=0.9,
        alpha=0.8,
    )

    viol = bt[bt["violation"] == 1]
    axes[1].scatter(
        viol["asof"],
        -viol["realized_1w_return"],
        color="red",
        s=25,
        zorder=5,
        label="violation",
    )

    axes[1].set_title("VaR vs realized loss over time")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

else:
    print(
        "Too few usable observations for a meaningful backtest; increase BACKTEST_N_DATES "
        "or widen the data window."
    )



## 4. Horizon repricing

Scenarios are mapped back into instrument values via:

- equity repricing from simulated one-week log returns,
- scenario-wise repricing of a synthetic semi-annual coupon Treasury note from the simulated key-rate curve.


In [ ]:
# Treasury note definition and current repricing
issue_date = pd.Timestamp(effective_date)
horizon_date = issue_date + BDay(HORIZON_BDAYS)
maturity_date = issue_date + pd.DateOffset(years=10)

coupon_schedule = generate_coupon_schedule(issue_date, maturity_date, freq=PAY_FREQ)

# Issue #8: the bond is priced off the full simulated curve. The tenor grid is taken from the
# active key-rate columns (ascending order), so the pricer is consistent with whatever tenors the
# rates block uses.
RATE_TENOR_YEARS = [TENOR_YEARS[t] for t in key_rates.columns]

current_bond_price = price_coupon_bond_from_curve(
    eval_date=issue_date,
    payment_dates=coupon_schedule,
    key_rate_series=current_key_rates,
    tenor_years=RATE_TENOR_YEARS,
    coupon_rate=COUPON_RATE,
    face_value=FACE_VALUE,
    freq=PAY_FREQ,
)

# Issue #10: clean/dirty decomposition. The DCF above returns the dirty (full) PV; accrued interest
# separates the clean price. At issue the note has just been struck, so accrued ~ 0.
accrued_at_issue = accrued_interest(issue_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)
accrued_at_horizon = accrued_interest(horizon_date, coupon_schedule, COUPON_RATE, FACE_VALUE, PAY_FREQ)

print(f"Synthetic Treasury note dirty price at issue: {current_bond_price:,.4f}")
print(f"  accrued at issue: {accrued_at_issue:,.4f}  ->  clean: {current_bond_price - accrued_at_issue:,.4f}")
print(f"Issue date   : {issue_date.date()}")
print(f"Horizon date : {pd.Timestamp(horizon_date).date()}")
print(f"Maturity date: {maturity_date.date()}")
print(
    "Pricing convention (declared approximation): FRED CMT par yields are linearly interpolated "
    "across tenor and treated as continuously-compounded zero rates. This conflates par-yield with "
    "zero-rate and bond-equivalent with continuous compounding (a small systematic bias acceptable "
    "for a prototype). P&L below is dirty-to-dirty, which already includes one-week carry / "
    "pull-to-par; accrued interest is reported only for the clean/dirty split."
)

bond_horizon_prices = key_rate_scenarios.apply(
    lambda row: price_coupon_bond_from_curve(
        eval_date=horizon_date,
        payment_dates=coupon_schedule,
        key_rate_series=row,
        tenor_years=RATE_TENOR_YEARS,
        coupon_rate=COUPON_RATE,
        face_value=FACE_VALUE,
        freq=PAY_FREQ,
    ),
    axis=1,
)
bond_horizon_prices.name = "UST_10Y_2pct"
bond_pnl = bond_horizon_prices - current_bond_price

print(f"\nMean dirty PV at horizon: {bond_horizon_prices.mean():,.4f} "
      f"(accrued at horizon: {accrued_at_horizon:,.4f})")
display(distribution_summary(bond_horizon_prices).to_frame("UST_10Y_2pct_horizon_value"))
display(loss_summary_from_pnl(bond_pnl).to_frame("UST_10Y_2pct_one_week_pnl"))

instrument_horizon_values = stock_price_scenarios.copy()
instrument_horizon_values["UST_10Y_2pct"] = bond_horizon_prices

current_values = current_prices.copy()
current_values["UST_10Y_2pct"] = current_bond_price

instrument_pnl = instrument_horizon_values.sub(current_values, axis=1)
instrument_returns = instrument_pnl.div(current_values, axis=1)

print("Horizon values:")
display(instrument_horizon_values.describe().T)

print("Instrument-level one-week P&L:")
display(instrument_pnl.describe().T)

print("Capital-normalized one-week return/loss metrics:")
display(pd.DataFrame({
    col: loss_summary_from_pnl(instrument_returns[col], alpha=CVAR_ALPHA)
    for col in instrument_returns.columns
}).T)


In [ ]:

# Selected instrument-level visualizations
plot_histogram(instrument_horizon_values[REPRESENTATIVE_EQUITY], f"Horizon-value distribution — {REPRESENTATIVE_EQUITY}", "Horizon value")
plot_histogram(instrument_pnl[REPRESENTATIVE_EQUITY], f"P&L distribution — {REPRESENTATIVE_EQUITY}", "One-week P&L")
plot_histogram(instrument_horizon_values["UST_10Y_2pct"], "Horizon-value distribution — UST_10Y_2pct", "Horizon value")
plot_histogram(instrument_pnl["UST_10Y_2pct"], "P&L distribution — UST_10Y_2pct", "One-week P&L")



## 5. Portfolio layer under capital-normalized scenario returns

Portfolio comparisons are performed on **one-week simple returns**, not raw dollar P&L per instrument unit. This keeps tail-risk comparisons economically interpretable when assets have different nominal prices.


In [ ]:
# Candidate portfolio weights in capital-normalized return space
all_assets = instrument_returns.columns.tolist()
equity_assets = TICKERS.copy()

equal_weight = pd.Series(1.0 / len(all_assets), index=all_assets)

equity_only = pd.Series(0.0, index=all_assets)
equity_only.loc[equity_assets] = 1.0 / len(equity_assets)

bond_heavy = pd.Series((1.0 - 0.60) / len(equity_assets), index=all_assets)
bond_heavy["UST_10Y_2pct"] = 0.60

inverse_vol = inverse_vol_weights_from_scenarios(instrument_returns)

# Issue #3: exact long-only min-CVaR via the Rockafellar-Uryasev LP (no more random search).
min_cvar_long_only, min_cvar_info = search_long_only_min_cvar_portfolio(
    instrument_returns=instrument_returns,
    alpha=CVAR_ALPHA,
    max_weight=CVAR_MAX_WEIGHT,
)

portfolio_weights = {
    "equal_weight": equal_weight,
    "equity_only": equity_only,
    "bond_heavy": bond_heavy,
    "inverse_vol": inverse_vol,
    "min_cvar_long_only": min_cvar_long_only,
}

portfolio_returns = {}
portfolio_metrics = {}
for name, w in portfolio_weights.items():
    port_ret, metrics = portfolio_metrics_from_instrument_returns(w, instrument_returns, alpha=CVAR_ALPHA)
    portfolio_returns[name] = port_ret
    portfolio_metrics[name] = metrics

print("Long-only min-CVaR LP diagnostics:")
display(min_cvar_info.to_frame("value"))

print("Portfolio weights:")
display(pd.DataFrame(portfolio_weights).T)

print("Portfolio one-week return/loss metrics:")
display(pd.DataFrame(portfolio_metrics).T.sort_values("ES_95_loss"))


In [ ]:
# Out-of-sample (fresh-draw) portfolio tail risk (issue #4)
# The min-CVaR weights are chosen to minimize ES on the in-sample scenario set, so the in-sample
# ES is optimistically biased (selection on noise). We simulate an INDEPENDENT scenario set with a
# different seed, reprice every instrument, and re-evaluate the SAME weights to get an honest ES.

sim_oos = simulate_joint_horizon(
    var_res=var_res,
    resid_hist=var_res.resid,
    last_state=drivers.iloc[-1].to_numpy(),
    equity_block=equity_block,
    rates_block=rates_block,
    current_prices=current_prices,
    current_key_rates=current_key_rates,
    horizon_bdays=HORIZON_BDAYS,
    n_scenarios=N_SCENARIOS,
    avg_block_length=BOOTSTRAP_EXPECTED_BLOCK,
    rng_seed=VALIDATION_SEED,
    daily_rf=current_daily_rf,
    resid_idio_std=equity_block.get("resid_idiosyncratic_std"),
)

bond_horizon_prices_oos = sim_oos["key_rate_scenarios"].apply(
    lambda row: price_coupon_bond_from_curve(
        eval_date=horizon_date,
        payment_dates=coupon_schedule,
        key_rate_series=row,
        tenor_years=RATE_TENOR_YEARS,
        coupon_rate=COUPON_RATE,
        face_value=FACE_VALUE,
        freq=PAY_FREQ,
    ),
    axis=1,
)

instrument_horizon_values_oos = sim_oos["stock_price_scenarios"].copy()
instrument_horizon_values_oos["UST_10Y_2pct"] = bond_horizon_prices_oos
instrument_returns_oos = instrument_horizon_values_oos.sub(current_values, axis=1).div(current_values, axis=1)

oos_rows = {}
for name, w in portfolio_weights.items():
    _, m_in = portfolio_metrics_from_instrument_returns(w, instrument_returns, alpha=CVAR_ALPHA)
    _, m_oos = portfolio_metrics_from_instrument_returns(w, instrument_returns_oos, alpha=CVAR_ALPHA)
    oos_rows[name] = {
        "VaR_95_in_sample": m_in["VaR_95_loss"],
        "VaR_95_out_of_sample": m_oos["VaR_95_loss"],
        "ES_95_in_sample": m_in["ES_95_loss"],
        "ES_95_out_of_sample": m_oos["ES_95_loss"],
    }

oos_table = pd.DataFrame(oos_rows).T
oos_table["ES_in_sample_optimism"] = oos_table["ES_95_out_of_sample"] - oos_table["ES_95_in_sample"]
print("In-sample vs out-of-sample one-week tail risk (independent scenario draw):")
display(oos_table.sort_values("ES_95_out_of_sample"))
print(
    "A positive 'ES_in_sample_optimism' for min_cvar_long_only is the selection-on-noise effect: "
    "the weights were fit to the in-sample tail, so their ES degrades out-of-sample. Policy "
    "portfolios (equal_weight, equity_only, bond_heavy) are not fit to the scenarios and should "
    "show only sampling noise between the two columns."
)


In [ ]:

# Portfolio-level distributions and weighted tail contributions
for name, port_ret in portfolio_returns.items():
    plot_histogram(port_ret, f"Portfolio one-week return distribution — {name}", "One-week portfolio return")

tail_table = tail_contribution_table(
    portfolio_return=portfolio_returns["min_cvar_long_only"],
    instrument_returns=instrument_returns,
    weights=portfolio_weights["min_cvar_long_only"],
    alpha=CVAR_ALPHA,
)

display(tail_table)


## 6. What changed (v5 → v6)

The v5 upgrades are retained:

1. **Broader US equity framing** instead of a narrow tech-only prototype.
2. **Modular ERP engines** with user-selectable statistical, proxy-based, and APT-style factor blocks.
3. A cleaner distinction between **risk-free curve dynamics**, **equity-premium generation**, and **idiosyncratic residual modelling**.

On top of that, **v6 applies the following methodological fixes** (structure unchanged):

1. **Validation suite expanded.** Beyond the single-name std check, the validation cell now compares simulated vs realized cross-equity correlation, equity-vs-rates correlation, and excess kurtosis; and a new **walk-forward VaR backtest** runs the **Kupiec POF** and **Christoffersen independence** tests plus a **PIT/rank histogram**.
2. **Walk-forward instead of a single as-of date.** The engine is refit on an expanding window across a panel of past as-of dates, and the VaR coverage is tested out-of-sample.
3. **Exact min-CVaR.** The random simplex search is replaced by the **Rockafellar-Uryasev linear program** (`scipy.optimize.linprog`), giving the true optimum, reproducibly.
4. **Out-of-sample portfolio tail risk.** The chosen min-CVaR weights are re-evaluated on an **independent scenario draw** to expose in-sample optimism.
5. **Vectorized simulator.** The double Python loop is replaced by batch matrix operations across scenarios (bit-identical to the loop given the same bootstrap indices).
6. **Unified risk-free.** Equity excess returns now use the **Fama-French daily RF**, consistent with `MKT_RF`, instead of `DGS2/252`.
7. **Additive FF factors.** `log1p` is dropped from the long-short spread factors (SMB/HML/MOM), which are treated as additive simple returns.
8. **Richer Treasury curve.** Ten tenors (1M-30Y) feed the rates PCA, so the level/slope/curvature decomposition is genuine and the bond repricing is curve-consistent.
9. **VAR(1) diagnostics.** A stability check (`is_stable`, root moduli) and a per-equation one-step **R²** quantify how much the VAR dynamics actually contribute.
10. **Declared bond assumptions + accrued interest.** The CMT-par-as-continuous-zero approximation is documented, and a clean/dirty (accrued) decomposition is reported.
11. **Idiosyncratic noise re-injected.** The residual variance dropped by the PCA compression is added back as iid noise so single-name tails are not understated.

This keeps the project a single reusable **research/GitHub scenario engine** while tightening its quantitative and methodological foundations.
